# Met Office UKV Model – AWS ASDI Demo

This notebook demonstrates how to use the `site_archive_aws.MOUKV` data accessor
to read Met Office UKV (UK Variable-Resolution Model) data from AWS S3.

## Dataset
The UKV is the Met Office's operational high-resolution deterministic model
covering the United Kingdom at approximately 1.5 km grid spacing.
It runs every hour and provides forecasts out to 36 hours.

The UKV uses a rotated-pole coordinate system; the accessor transparently
renames `grid_latitude`/`grid_longitude` to standard `latitude`/`longitude`.

## Requirements
```
pip install pyearthtools-archive-aws
```


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import site_archive_aws
from site_archive_aws import MOUKV

print(f"site_archive_aws version: {site_archive_aws.__version__}")

## 1. Configure the Accessor

In [ ]:
# The UKV runs every hour; choose any hour
QUERY_TIME = "2023-06-01T06:00"

# Create accessor for 2-m temperature
accessor = MOUKV("2t", anon=True)
print(accessor)

## 2. Load 2-m Temperature

In [ ]:
ds = accessor[QUERY_TIME]
print(ds)

# Convert K → °C
t2m = ds["air_temperature"].squeeze() - 273.15

## 3. Map of 2-m Temperature over the UK

In [ ]:
fig, ax = plt.subplots(
    figsize=(8, 10),
    subplot_kw={"projection": ccrs.PlateCarree()},
)

im = ax.contourf(
    t2m["longitude"],
    t2m["latitude"],
    t2m.values,
    levels=np.linspace(-5, 30, 36),
    cmap="RdBu_r",
    transform=ccrs.PlateCarree(),
)

ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.add_feature(cfeature.RIVERS.with_scale("10m"), linewidth=0.3, alpha=0.5)
ax.gridlines(draw_labels=True, linewidth=0.3)

# Zoom to UK
ax.set_extent([-10, 3, 49, 61], crs=ccrs.PlateCarree())

plt.colorbar(im, ax=ax, label="2-m Temperature (°C)", shrink=0.7)
ax.set_title(
    f"Met Office UKV – 2-m Temperature\n{QUERY_TIME} (UTC)",
    fontsize=13,
)
plt.tight_layout()
plt.savefig("ukv_2t.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to ukv_2t.png")

## 4. Total Precipitation

In [ ]:
# Load total precipitation
acc_precip = MOUKV("tp", anon=True)
ds_precip = acc_precip[QUERY_TIME]
precip = ds_precip["stratiform_rainfall_amount"].squeeze()  # kg/m² ≈ mm

fig, ax = plt.subplots(
    figsize=(8, 10),
    subplot_kw={"projection": ccrs.PlateCarree()},
)

im = ax.contourf(
    precip["longitude"],
    precip["latitude"],
    precip.values,
    levels=[0, 0.1, 0.5, 1, 2, 4, 8, 16, 32, 64],
    cmap="Blues",
    transform=ccrs.PlateCarree(),
)

ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.set_extent([-10, 3, 49, 61], crs=ccrs.PlateCarree())
plt.colorbar(im, ax=ax, label="Total Precipitation (mm)", shrink=0.7)
ax.set_title(
    f"Met Office UKV – Total Precipitation\n{QUERY_TIME} (UTC)",
    fontsize=13,
)
plt.tight_layout()
plt.savefig("ukv_precip.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to ukv_precip.png")

## 5. Summary

In this notebook we:
1. Loaded UKV 2-m temperature and produced a UK regional map.
2. Loaded UKV total precipitation and visualised it.

See `scripts/demo_mo_ukv.py` for the command-line equivalent.